In [2]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

df = pd.read_csv("window_features_clean.csv")
FEATURES = ["hr_mean", "hr_std", "hr_slope", "spo2_mean", "spo2_std", "spo2_slope", "map_mean", "map_std", "map_slope"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["caseid"]))

train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]
X_train, X_test = train_df[FEATURES], test_df[FEATURES]

In [3]:
print("Training XGBoost: Tachycardia")
y_train_tach = train_df["tachycardia"]
y_test_tach = test_df["tachycardia"]

xgb_tach = xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42, eval_metric='auc')
xgb_tach.fit(X_train, y_train_tach)

prob_tach = xgb_tach.predict_proba(X_test)[:, 1]
print(f"Tachycardia XGB AUROC: {roc_auc_score(y_test_tach, prob_tach):.4f}")

Training XGBoost: Tachycardia
Tachycardia XGB AUROC: 0.7735


In [4]:
print("Training XGBoost: Hypotension")
y_train_hypo = train_df["hypotension"]
y_test_hypo = test_df["hypotension"]

xgb_hypo = xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42, eval_metric='auc')
xgb_hypo.fit(X_train, y_train_hypo)

prob_hypo = xgb_hypo.predict_proba(X_test)[:, 1]
print(f"Hypotension XGB AUROC: {roc_auc_score(y_test_hypo, prob_hypo):.4f}")

Training XGBoost: Hypotension
Hypotension XGB AUROC: 0.7214


In [5]:
print("Training XGBoost: Hypoxia")
y_train_hypox = train_df["hypoxia"]
y_test_hypox = test_df["hypoxia"]

imbalance_ratio = (len(y_train_hypox) - y_train_hypox.sum()) / y_train_hypox.sum()

xgb_hypox = xgb.XGBClassifier(n_estimators=500, learning_rate=0.01, max_depth=5, scale_pos_weight=imbalance_ratio, random_state=42, eval_metric='auc')
xgb_hypox.fit(X_train, y_train_hypox)

prob_hypox = xgb_hypox.predict_proba(X_test)[:, 1]
print(f"Hypoxia XGB AUROC: {roc_auc_score(y_test_hypox, prob_hypox):.4f}")

Training XGBoost: Hypoxia
Hypoxia XGB AUROC: 0.6856
